## 1) Libraries Installation
##### The cell below is to help you keep track the libraries used and install them quickly.
##### Ensure the correct library names are used, and follow the syntax: **%pip install PACKAGE_NAME**.

In [ ]:
library(readr)
library(quantmod)
library(dplyr)
library(tseries)
library(xts)
library(forecast)
library(lmtest)
library(vars)
library(e1071)
library(xgboost)
library(randomForest)

ERROR: Error in library(quantmod): there is no package called ‘quantmod’


## 2) Main Section for Code
### Preparing dataset

In [ ]:
# read csv
quarterly = read.csv('2025-01.csv')
quarterly <- quarterly[-c(1,2),]

quarterly$t_spread <- quarterly$GS10 - quarterly$FEDFUND
# Create a new column 'consumerx_rate' with the percentage change from the previous row

Preparing GDP growth rate

In [ ]:
quarterly <- quarterly %>%
  mutate(sasdate = as.Date(sasdate, format = "%m/%d/%Y"))
GDP <- xts(quarterly$GDPC1, quarterly$sasdate)
GDPGrowth <- xts(400 * log(GDP/lag(GDP)))
quarterly$gdp_growth <- ts(GDPGrowth)
# compute logarithms, annual growth rates and 1st lag of growth rates
quants <- function(series) {
  s <- series
  return(
    data.frame("Level" = s,
               "Logarithm" = log(s),
               "AnnualGrowthRate" = 400 * log(s / lag(s)),
               "1stLagAnnualGrowthRate" = lag(400 * log(s / lag(s))))
    )
}

quants(GDP["2023-06::2024-12"])
GDPGrowth

ERROR: Error in quarterly %>% mutate(sasdate = as.Date(sasdate, format = "%m/%d/%Y")): could not find function "%>%"


Plot GDPGrowth

In [ ]:
plot(GDPGrowth)


ERROR: Error: object 'GDPGrowth' not found


Testing stationary

In [ ]:
adf.test(GDPGrowth["1960::2024"])
# this proves our gdp growth is stationary


ar.ols(GDPGrowth["1960::2024"],
       order.max = 100,
       demean = F,
       intercept = T)

Warning message in adf.test(GDPGrowth["1960::2024"]):
“p-value smaller than printed p-value”



	Augmented Dickey-Fuller Test

data:  GDPGrowth["1960::2024"]
Dickey-Fuller = -6.3814, Lag order = 6, p-value = 0.01
alternative hypothesis: stationary



Call:
ar.ols(x = GDPGrowth["1960::2024"], order.max = 100, demean = F,     intercept = T)

Coefficients:
     1       2  
0.0350  0.0992  

Intercept: 2.559 (0.3645) 

Order selected 2  sigma^2 estimated as  17.75

Looking for useful predictors -> granger causality tests

In [ ]:
var_xts <- xts(quarterly$CONSUMERx, order.by = quarterly$sasdate)
var_aligned <- var_xts["1960-06-01/"]
GDPGrowth_aligned <- GDPGrowth["1960-06-01/"]

# Ensure the column name is set
colnames(var_aligned) <- "var"

# Convert to numeric vectors
var_aligned <- as.numeric(var_aligned$var)
GDPGrowth_data <- as.numeric(GDPGrowth_aligned$x)
grangertest(GDPGrowth_data ~ var_aligned, order = 2)

# sig:
# CLAIMSx: 2.863e-05 ***
# ACOGNOx: 0.01719 * (orders of consumer goods)
# PERMIT:  4.671e-06 *** (private housing)
# UMCSENT:  0.02649 * (consumer sentiments)
# AMDMNOx: 0.006744 **
# t_spread: 0.001264 **


# insig:
# ANDENOx: 0.2995 (Real Value of Manufacturers' New Orders for Capital Goods: Nondefense Capital Goods Industries (Million of 2017 Dollars), deflated by Core PCE)
# AWHMAN: 0.2755


Training model basic

In [ ]:
# CLAIMSx: 2.863e-05 ***
# ACOGNOx: 0.01719 * (orders of consumer goods)
# PERMIT:  4.671e-06 *** (private housing)
# UMCSENT:  0.02649 * (consumer sentiments)
# AMDMNOx: 0.006744 **
# t_spread: 0.001264 **

claims <- xts(quarterly$CLAIMSx, order.by = quarterly$sasdate)
acognox <- xts(quarterly$ACOGNOx, order.by = quarterly$sasdate)

train_up_to_2024Q3 <- GDPGrowth["1960/2024-09-01"]

# Split into train and test
train <-train_up_to_2024Q3
# test <- GDPGrowth["2000/2024"]
test <- GDPGrowth["2024-12-01"]

# Fit the model on training set
model <- ar.ols(train,
       order.max = 50,
       demean = F,
       intercept = T,
       xreg = train[, c(claims,acognox)])
model
# Forecast on the test set
forecasted <- forecast(model, h = length(test))
# plot(forecasted)

# Extract the forecasted values
forecasted_values <- forecasted$mean

# Getting RMSFE
# Extract the actual values (from the test set)
# Assuming 'test' is the actual data for comparison
actual_values <- test
forecasted_xts <- xts(forecasted_values, order.by = index(actual_values))

# Compute squared errors.  Now xts objects are being subtracted.
squared_errors <- (actual_values - forecasted_xts)^2


# Compute RMSFE (handle potential NA values)
rmsfe <- sqrt(mean(squared_errors, na.rm = TRUE))
print(paste("RMSFE:", rmsfe))



# ADL model

In [ ]:
library(dynlm)
quarterly_xts <- xts(quarterly[,-1], order.by = quarterly$sasdate)
CLAIMSx <- quarterly_xts$CLAIMSx
ACOGNOx <- quarterly_xts$ACOGNOx

ACOGNOx_ts <- ts(ACOGNOx, start = c(1959, 1), frequency = 4)
CLAIMSx_ts <- ts(CLAIMSx, start = c(1959, 1), frequency = 4)
GDPGrowth_ts <- ts(GDPGrowth,start = c(1959, 1), end=c(2024,4), frequency = 4)


ADLdata <- ts.union(GDPGrowth_ts, CLAIMSx_ts,ACOGNOx_ts )
# ADLdata

# Subset the time series data for the period 1960 to 2024 <- change these values for subset
GDPGrowth_subset <- window(GDPGrowth_ts, start = c(1960, 1), end = c(2024, 3))
ACOGNOx_subset <- window(ACOGNOx_ts, start = c(1960, 1), end = c(2024, 3))
CLAIMSx_subset <- window(CLAIMSx_ts, start = c(1960, 1), end = c(2024, 3))

# Combine the subsetted data into a new object (if needed)
ADLdata_subset <- ts.union(GDPGrowth_subset, CLAIMSx_subset, ACOGNOx_subset)

# ADLdata_subset

# Fit the model using the subsetted data (change the L and numbers to adjust lags)
model_dynlm <- dynlm(GDPGrowth_subset ~ L(GDPGrowth_subset) + L(GDPGrowth_subset, 2) +
                     L(ACOGNOx_subset) + L(CLAIMSx_subset))



summary(model_dynlm)
#coeftest(model_dynlm, vcov. = sandwich)

ERROR: Error in library(dynlm): there is no package called ‘dynlm’


Plotting predictions

#### Remember to rename your file name to **NUS_DSESC_DATABUSTERS_XX.ipynb** and ensure that it can run successfully. Good luck and have fun!

# ARIMA simple model


In [ ]:
```{r}
library(readr)
library(quantmod)
library(dplyr)
library(ggplot2)
library(gridExtra)
library(corrplot)
library(tseries)
library(xts)
library(forecast)
library(zoo)
library(lmtest)
library(vars)
library(dynlm)
```

```{r}
quarterly = read.csv('2025-01.csv')
quarterly <- quarterly[-c(1,2),] # get rid of first 2 redundant rows
```

```{r}
# Change sasdate to date and datetime format
quarterly <- quarterly %>%
  mutate(sasdate = as.Date(sasdate, format = "%m/%d/%Y"))
# Calculate annual GDP growth rate manually
quarterly$gdp_growth <- 400 * log(quarterly$GDPC1 / lag(quarterly$GDPC1))
GDP <- xts(quarterly$GDPC1, quarterly$sasdate)
GDPGrowth <- xts(400 * log(GDP/lag(GDP)))


# Calculate logarithms, annual growth rates and 1st lag of growth rates
quants <- function(series) {
  data.frame(
    Level = series,
    Logarithm = log(series),
    AnnualGrowthRate = 400 * log(series / lag(series)),
    FirstLagAnnualGrowthRate = lag(400 * log(series / lag(series)))
  )
}
quarterly$t_spread <- quarterly$GS10 - quarterly$FEDFUND
quarterly$corpyield_spread <- quarterly$BAA - quarterly$AAA
```

```{r}
UNRATE_xts <- xts(quarterly$UNRATE, order.by = quarterly$sasdate)
CLAIMSx_xts <- xts(quarterly$CLAIMSx, order.by = quarterly$sasdate)
PERMIT_xts <- xts(quarterly$PERMIT, order.by = quarterly$sasdate)
ACOGNOx_xts <- xts(quarterly$ACOGNOx, order.by = quarterly$sasdate)
corpyield_xts <- xts(quarterly$corpyield_spread, order.by = quarterly$sasdate)
t_spread_xts <- xts(quarterly$t_spread, order.by = quarterly$sasdate)
UMCSENTx_xts <- xts(quarterly$UMCSENTx, order.by = quarterly$sasdate)
SP500_xts<- xts(quarterly$`S.P.500`, order.by = quarterly$sasdate)
OPHMFG_xts <- xts(quarterly$OPHMFG, order.by = quarterly$sasdate)
OPHNFB_xts <- xts(quarterly$OPHNFB, order.by = quarterly$sasdate)
OPHPBS_xts <- xts(quarterly$OPHPBS, order.by = quarterly$sasdate)
```

```{r}
adf.test(GDPGrowth["1960::2024"])
adf.test(t_spread_xts["1960::2024"])
adf.test(corpyield_xts["1960::2024"])
adf.test(CLAIMSx_xts["1967::2024"])
adf.test(PERMIT_xts["1967::2024"])
adf.test(ACOGNOx_xts["1992::2024"]) #non-stationary
ACOGNOx_diff <- diff(ACOGNOx_xts)
adf.test(ACOGNOx_diff["1992-06::2024"])
adf.test(UMCSENTx_xts["1967::2024"]) #non-stationary
UMCSENTx_diff <- diff(UMCSENTx_xts)
adf.test(UMCSENTx_diff["1967::2024"])
adf.test(SP500_xts["1967::2024"])#non-stationary
SP500_diff <- diff(SP500_xts)
adf.test(SP500_diff["1967::2024"])
adf.test(OPHMFG_xts["1987::2024-09"])
adf.test(OPHNFB_xts["1967::2024-09"])
adf.test(OPHPBS_xts["1967::2024-09"])
OPHMFG_diff <- diff(OPHMFG_xts)
OPHNFB_diff <- diff(OPHNFB_xts)
OPHPBS_diff <- diff(OPHPBS_xts)
adf.test(OPHMFG_diff["1987-06::2024-09"])
adf.test(OPHNFB_diff["1967::2024-09"])
adf.test(OPHPBS_diff["1967::2024-09"])


#now all stationary
quarterly$ACOGNOx <- c(NA, diff(quarterly$ACOGNOx))
quarterly$UMCSENTx <- c(NA, diff(quarterly$UMCSENTx))
quarterly$S.P.500 <- c(NA, diff(quarterly$S.P.500))
quarterly$OPHMFG <- c(NA, diff(quarterly$OPHMFG))
quarterly$OPHNFB<- c(NA, diff(quarterly$OPHNFB))
quarterly$OPHPBS<- c(NA, diff(quarterly$OPHPBS))
```

```{r}
#Preliminary analysis of GDP growth rate
autoplot(GDPGrowth) + ggtitle("Growth Rates")
```

```{r}
# Split the GDPGrowth data into training and testing sets
train <- GDPGrowth["1960/2011"]
test  <- GDPGrowth["2012/2024"]

# Convert the xts objects to time series objects for modeling
train_ts <- ts(as.numeric(train), start = c(1960, 1), frequency = 4)
test_ts  <- ts(as.numeric(test),  start = c(2017, 1), frequency = 4)


# Fit an ARIMA model on the training set
fit_arima_train <- auto.arima(train_ts)
summary(fit_arima_train)

# Forecast over the horizon equal to the length of the test set
forecast_horizon <- length(test_ts)
fcst_train <- forecast(fit_arima_train, h = forecast_horizon)

# Plot the forecast along with the actual test data
library(zoo)  # for as.yearqtr()

# Convert test_ts to a data frame with a Date column representing quarters
test_df <- data.frame(
  Date = as.Date(as.yearqtr(time(test_ts))),
  Test = as.numeric(test_ts)
)

# Now, plot the forecast and add the test data as a red line
autoplot(fcst_train) +
  geom_line(data = test_df, aes(x = Date, y = Test), color = "red") +
  ggtitle("GDP Growth Rate Forecast vs Test Data") +
  xlab("Year") + ylab("GDP Growth Rate")
rmsfe <- sqrt(mean((as.numeric(fcst_train$mean) - as.numeric(test_ts))^2))
rmsfe
```
```{r}
fit_arima = auto.arima(GDPGrowth)
fit_arima
fcst <- forecast(fit_arima, h=3)
autoplot(fcst)
print(fcst)

ERROR: Error in parse(text = input): attempt to use zero-length variable name


In [ ]:
```{r}
# ---------------------------
# 1. Prepare Exogenous Variables
# ---------------------------
exog_vars <- c("OPHNFB", "CLAIMSx", "PERMIT",
               "ACOGNOx", "corpyield_spread", "t_spread", "UMCSENTx", "S.P.500")
# Create an xts object for exogenous variables using the date column
exog_xts <- xts(quarterly[, exog_vars], order.by = quarterly$sasdate)

# ---------------------------
# 2. Define Training and Testing Periods for GDPGrowth
# ---------------------------
# For this example, we assume that GDPGrowth data spans from 1992 Q1 to 2024 Q4.
# We choose the training period as 1992-2011 and the test period as 2012-2024.
train <- GDPGrowth["1992-01-01/2011-12-31"]
test  <- GDPGrowth["2012-01-01/2024-12-31"]

# Convert the xts subsets into ts objects.
# IMPORTANT: Set the start dates to reflect the actual periods.
train_ts <- ts(as.numeric(train), start = c(1992, 1), frequency = 4)
test_ts  <- ts(as.numeric(test), start = c(2012, 1), frequency = 4)

cat("Training period: ", start(train_ts), "to", end(train_ts), "\n")
cat("Test period: ", start(test_ts), "to", end(test_ts), "\n")

# ---------------------------
# 3. Align Exogenous Variables with the Training and Testing Periods
# ---------------------------
# Subset exogenous variables to the same date ranges
exog_train <- exog_xts["1992-01-01/2011-12-31"]
exog_test  <- exog_xts["2012-01-01/2024-12-31"]

# Convert these subsets to ts objects with matching start dates
exog_train_ts <- ts(as.matrix(exog_train), start = c(1992, 1), frequency = 4)
exog_test_ts  <- ts(as.matrix(exog_test), start = c(2012, 1), frequency = 4)

# ---------------------------
# 4. Fit the ARIMAX Model Using the Training Data
# ---------------------------
# Combine GDPGrowth (target) with the exogenous regressors and remove any NA rows
combined_train <- na.omit(cbind(train_ts, exog_train_ts))
train_ts_clean <- combined_train[, 1]
exog_train_ts_clean <- combined_train[, -1]

# Fit an ARIMAX model (ARIMA with exogenous variables)
fit_arimax_train <- auto.arima(train_ts_clean,
                               xreg = exog_train_ts_clean,
                               stepwise = FALSE,
                               approximation = FALSE)
print(fit_arimax_train)

# ---------------------------
# 5. Forecast Using the ARIMAX Model
# ---------------------------
# Check that the dimensions of exog_test_ts match the length of test_ts
if(nrow(exog_test_ts) != length(test_ts)){
  stop("Mismatch between test_ts and exog_test_ts dimensions. Please check the subsetting.")
}

# Forecast over the test period using the exogenous regressors
fcst_arimax <- forecast(fit_arimax_train, xreg = exog_test_ts, h = length(test_ts))

# ---------------------------
# 6. Plot Forecast vs. Actual Data
# ---------------------------
library(zoo)  # for as.yearqtr conversion
test_df <- data.frame(
  Date = as.Date(as.yearqtr(time(test_ts))),
  TestData = as.numeric(test_ts)
)

autoplot(fcst_arimax, series = "Forecast") +
  geom_line(data = test_df, aes(x = Date, y = TestData, color = "Actual"), size = 1) +
  ggtitle("ARIMAX Forecast vs Test Data") +
  xlab("Year") +
  ylab("GDP Growth") +
  scale_colour_manual(name = "Series", values = c("Forecast" = "blue", "Actual" = "red"))

# ---------------------------
# 7. Compute Forecast Accuracy (RMSFE)
# ---------------------------
forecast_values_arimax <- fcst_arimax$mean
actual_values_arimax <- test_ts
rmsfe_arimax <- sqrt(mean((actual_values_arimax - forecast_values_arimax)^2, na.rm = TRUE))
cat("ARIMAX RMSFE:", rmsfe_arimax, "\n")

```